# Decile Analysis for Customer Segmentation and Sales Contribution

In [ ]:
# Purchase history logs for each customer.（Dummy data）
# Including data showing multiple purchases by the same customer, 
# and data containing invalid negative amounts.
sales_logs = [
    {"customer_id": "C001", "amount": 5000},
    {"customer_id": "C002", "amount": 12000},
    {"customer_id": "C003", "amount": 3000},
    {"customer_id": "C001", "amount": 4000}, # Second purchase of C001
    {"customer_id": "C004", "amount": 15000},
    {"customer_id": "C005", "amount": -2000}, # invalid data
    {"customer_id": "C006", "amount": 8000},
    {"customer_id": "C007", "amount": 25000},
    {"customer_id": "C008", "amount": 1000},
    {"customer_id": "C009", "amount": 6000},
    {"customer_id": "C010", "amount": 9000},
    {"customer_id": "C011", "amount": 0},    # invalid data (zero amount excluded)
    {"customer_id": "C012", "amount": 11000},
]

def get_users(logs):
    """
    Exclude amounts that are 0 or less, and calculate the total purchase amount for each customer.

    Args:
        logs (list): Sales logs. Each log is a dictionary containing customer_id and amount.
    
    Returns:
        list: A list of dictionaries containing customer_id and total_amount.
    """
    totals = {}
    for log in logs:
        if log["amount"] > 0:
            customer_id = log["customer_id"]
            if customer_id in totals:
                totals[customer_id] += log["amount"]
            else:
                totals[customer_id] = log["amount"]

    users = [
        {"customer_id": cid, "total_amount": amount}
        for cid, amount in totals.items()
    ]
            
    return users

def decile_division(logs):
    """
    Sort customers in descending order of total purchase amount (highest to lowest),
    and divide the sorted customer list into 10 equal parts (Decile 1 to Decile 10).

    Args:
        logs(list): Sales logs. Each log is a dictionary containing customer_id and total_amount.
    
    Returns:
        list: A list of lists, where each sublist represents a decile group
        containing dictionaries with customer_id and total_amount.
    """
    users = get_users(logs)
    sorted_users = sorted(users, key=lambda x: x["total_amount"], reverse=True)

    n = len(sorted_users)
    deciles = []
    for i in range(10):
        start = i * n // 10
        end = (i + 1) * n // 10
        decile_group = sorted_users[start:end]
        deciles.append(decile_group)

    return deciles

def aggregate_deciles(logs):
    """
    Calculate the total purchase amount for that group for each decile group.

    Args:
        logs(list): Sales logs Each log is a dictionary containing customer_id and total_amount.

    Returns:
        list: A list of dictionaries containing decile, customer_count, and total_amount.
    """
    deciles = decile_division(logs)
    aggregate = []
    for i, decile in enumerate(deciles):
        total_amount = sum(user["total_amount"] for user in decile)

        aggregate.append({
            "decile": i + 1,
            "customer_count": len(decile),
            "total_amount": total_amount
        })
    
    return aggregate

def calculate_ratio(logs):
    """
    Calculate the sales composition ratio of each decile relative to the total sales,
    and the cumulative sales ratio from the group.
    
    Args:
        logs(list): Sales logs Each log is a dictionary containing customer_id and total_amount.
        
    Returns:
        list: A list of dictionaries containing decile, customer_count, total_amount,
        composition_ratio, and cumulative_ratio.
    """
    aggregate = aggregate_deciles(logs)
    total_sales = sum(d["total_amount"] for d in aggregate)

    current_ratio = 0
    for row in aggregate:
        composition_ratio = row["total_amount"] / total_sales * 100
        row["composition_ratio"] = composition_ratio
        cumulative_ratio = composition_ratio + current_ratio
        row["cumulative_ratio"] = cumulative_ratio
        current_ratio += composition_ratio
    
    return aggregate

def print_decile_summary(decile_data):
    """
    Print the total purchase amount, composition ratio, and cumulative ratio for each decile.

    Args:
        decile_data(list): A list of dictionaries containing decile, customer_count, total_amount,
        composition_ratio, and cumulative_ratio.
    """
    for data in decile_data:
        decile = data["decile"]
        total = data["total_amount"]
        com_ratio = data["composition_ratio"]
        cum_ratio = data["cumulative_ratio"]

        print(f"{decile:<6} | {total:15} | {com_ratio:16.1f}% | {cum_ratio:>15.1f}%")

print(f"decile | purchase amount | composition ratio | cumulative ratio")
result = calculate_ratio(sales_logs)
print_decile_summary(result)


decile | purchase amount | composition ratio | cumulative ratio
1      |           25000 |             25.3% |            25.3%
2      |           15000 |             15.2% |            40.4%
3      |           12000 |             12.1% |            52.5%
4      |           11000 |             11.1% |            63.6%
5      |            9000 |              9.1% |            72.7%
6      |            9000 |              9.1% |            81.8%
7      |            8000 |              8.1% |            89.9%
8      |            6000 |              6.1% |            96.0%
9      |            3000 |              3.0% |            99.0%
10     |            1000 |              1.0% |           100.0%


## What I learned

In this notebook, I practiced a basic decile analysis using dummy purchase history data.

First, I removed invalid purchase records where the amount was 0 or less.
Then, I aggregated multiple purchases by the same customer into a single total purchase amount.
After that, I sorted customers by total purcase amount in descending order and divided them into 10 decile groups.

I also calculated two important ratios:
・composition_ratio: the percentage of total sales generated by each decile
・cumulative_ratio: the cumulative percentage of sales from the top decile to the current decile

Through this exercise, I learned that decile analysis can be used to understand how much sales are cocentrated amoung high-value customers.

## Notes

In this dummy dataset, there are exactly 10 valid customers, so each decile contains one customer. In a real dataset, each decile would usually contain many customers, and the results would be more meaningful.

Also, invalid data such as negative amounts or zero amounts should be handled before performing customer analysis.

## Next step

Next, I want to visualize the decile results using a bar chart or line chart.
I also wanto to try the same analysis with a larger dataset and compare the sales concentration among customer groups.